In [9]:
import pandas as pd
import json
import re
import os

print("Libraries imported successfully")

Libraries imported successfully


In [10]:
recruiter_data = pd.read_csv("recruiter.csv")

recruiter_data

,candidate_id,name,email,phone,company,title
0,1,Rahul Sharma,rahul@gmail.com,9876543210,ABC Ltd,Developer
1,2,Amit Verma,amit@gmail.com,9999999999,XYZ Ltd,Software Engineer
2,3,Rahul S.,rahul@gmail.com,98765-43210,DEF Tech,Python Developer
3,4,Priya Singh,priya@gmail.com,8888888888,Tech Corp,Data Analyst


In [11]:
os.makedirs("output", exist_ok=True)

print("Output folder ready")

Output folder ready


In [12]:
with open("resumes.json") as f:
    resume_data = json.load(f)

resume_data

[{'candidate_id': '001',
  'name': 'Rahul Sharma',
  'email': 'rahul@gmail.com',
  'skills': ['Python', 'python programming', 'SQL'],
  'experience': '3 years',
  'education': 'B.Tech'},
 {'candidate_id': '002',
  'name': 'Amit Verma',
  'email': 'amit@gmail.com',
  'skills': ['Java', 'SQL'],
  'experience': '2 years',
  'education': 'BCA'},
 {'candidate_id': '004',
  'name': 'Priya Singh',
  'email': 'priya@gmail.com',
  'skills': ['Excel', 'SQL', 'Data Analysis'],
  'experience': '1 year',
  'education': 'MBA'}]

In [13]:
# Load CSV source
recruiter_df = pd.read_csv("recruiter.csv")


# Load resume source
with open("resumes.json") as f:
    resumes = json.load(f)


print("Recruiter records:", len(recruiter_df))
print("Resume records:", len(resumes))


display(recruiter_df)

Recruiter records: 4
Resume records: 3


,candidate_id,name,email,phone,company,title
0,1,Rahul Sharma,rahul@gmail.com,9876543210,ABC Ltd,Developer
1,2,Amit Verma,amit@gmail.com,9999999999,XYZ Ltd,Software Engineer
2,3,Rahul S.,rahul@gmail.com,98765-43210,DEF Tech,Python Developer
3,4,Priya Singh,priya@gmail.com,8888888888,Tech Corp,Data Analyst


In [14]:
def normalize_phone(phone):

    phone = re.sub(r'\D', '', str(phone))

    if len(phone) == 10:
        return "+91-" + phone

    return phone



def normalize_skill(skill):

    skill = skill.lower().strip()


    mapping = {
        "python programming":"Python",
        "python":"Python",
        "py":"Python",
        "sql":"SQL",
        "java":"Java",
        "excel":"Excel",
        "data analysis":"Data Analysis"
    }


    return mapping.get(skill, skill.title())



print(normalize_phone("98765-43210"))
print(normalize_skill("python programming"))

+91-9876543210
Python


In [15]:
def find_resume(email):

    for resume in resumes:

        if resume["email"] == email:
            return resume

    return None

In [16]:
final_profiles = []


for index,row in recruiter_df.iterrows():

    profile = {}


    profile["candidate_id"] = row["candidate_id"]

    profile["full_name"] = row["name"]


    profile["emails"] = [
        row["email"]
    ]


    profile["phones"] = [
        normalize_phone(row["phone"])
    ]


    resume = find_resume(row["email"])


    skills = []


    if resume:

        for skill in resume["skills"]:

            clean = normalize_skill(skill)

            if clean not in skills:
                skills.append(clean)



        profile["experience"] = [
            resume["experience"]
        ]


        profile["education"] = [
            resume["education"]
        ]

    else:

        profile["experience"] = []
        profile["education"] = []



    profile["skills"] = skills


    profile["provenance"] = [
        "recruiter.csv",
        "resumes.json"
    ]


    profile["confidence_score"] = 0.95



    final_profiles.append(profile)



final_profiles

[{'candidate_id': 1,
  'full_name': 'Rahul Sharma',
  'emails': ['rahul@gmail.com'],
  'phones': ['+91-9876543210'],
  'experience': ['3 years'],
  'education': ['B.Tech'],
  'skills': ['Python', 'SQL'],
  'provenance': ['recruiter.csv', 'resumes.json'],
  'confidence_score': 0.95},
 {'candidate_id': 2,
  'full_name': 'Amit Verma',
  'emails': ['amit@gmail.com'],
  'phones': ['+91-9999999999'],
  'experience': ['2 years'],
  'education': ['BCA'],
  'skills': ['Java', 'SQL'],
  'provenance': ['recruiter.csv', 'resumes.json'],
  'confidence_score': 0.95},
 {'candidate_id': 3,
  'full_name': 'Rahul S.',
  'emails': ['rahul@gmail.com'],
  'phones': ['+91-9876543210'],
  'experience': ['3 years'],
  'education': ['B.Tech'],
  'skills': ['Python', 'SQL'],
  'provenance': ['recruiter.csv', 'resumes.json'],
  'confidence_score': 0.95},
 {'candidate_id': 4,
  'full_name': 'Priya Singh',
  'emails': ['priya@gmail.com'],
  'phones': ['+91-8888888888'],
  'experience': ['1 year'],
  'education': [

In [18]:
merged_profiles = {}

for profile in final_profiles:

    email = profile["emails"][0]

    if email not in merged_profiles:
        merged_profiles[email] = profile

    else:
        existing = merged_profiles[email]

        # Merge skills
        existing["skills"] = list(
            set(existing["skills"] + profile["skills"])
        )

        # Merge phones
        existing["phones"] = list(
            set(existing["phones"] + profile["phones"])
        )

        # Increase confidence because multiple sources matched
        existing["confidence_score"] = 0.99


final_profiles = list(merged_profiles.values())

print("Final candidate count:", len(final_profiles))

Final candidate count: 3


In [19]:
config = {
    "fields": [
        "candidate_id",
        "full_name",
        "emails",
        "phones",
        "skills",
        "experience",
        "education",
        "confidence_score"
    ]
}

In [20]:
projected_profiles = []

for profile in final_profiles:

    projected = {}

    for field in config["fields"]:

        projected[field] = profile.get(field, None)

    projected_profiles.append(projected)

projected_profiles

[{'candidate_id': 1,
  'full_name': 'Rahul Sharma',
  'emails': ['rahul@gmail.com'],
  'phones': ['+91-9876543210'],
  'skills': ['Python', 'SQL'],
  'experience': ['3 years'],
  'education': ['B.Tech'],
  'confidence_score': 0.99},
 {'candidate_id': 2,
  'full_name': 'Amit Verma',
  'emails': ['amit@gmail.com'],
  'phones': ['+91-9999999999'],
  'skills': ['Java', 'SQL'],
  'experience': ['2 years'],
  'education': ['BCA'],
  'confidence_score': 0.95},
 {'candidate_id': 4,
  'full_name': 'Priya Singh',
  'emails': ['priya@gmail.com'],
  'phones': ['+91-8888888888'],
  'skills': ['Excel', 'SQL', 'Data Analysis'],
  'experience': ['1 year'],
  'education': ['MBA'],
  'confidence_score': 0.95}]

In [21]:
import json

with open("output/final_candidates.json", "w") as f:
    json.dump(projected_profiles, f, indent=4)

print("Output saved successfully")

Output saved successfully


In [22]:
with open("output/final_candidates.json") as f:
    result = json.load(f)

result

[{'candidate_id': 1,
  'full_name': 'Rahul Sharma',
  'emails': ['rahul@gmail.com'],
  'phones': ['+91-9876543210'],
  'skills': ['Python', 'SQL'],
  'experience': ['3 years'],
  'education': ['B.Tech'],
  'confidence_score': 0.99},
 {'candidate_id': 2,
  'full_name': 'Amit Verma',
  'emails': ['amit@gmail.com'],
  'phones': ['+91-9999999999'],
  'skills': ['Java', 'SQL'],
  'experience': ['2 years'],
  'education': ['BCA'],
  'confidence_score': 0.95},
 {'candidate_id': 4,
  'full_name': 'Priya Singh',
  'emails': ['priya@gmail.com'],
  'phones': ['+91-8888888888'],
  'skills': ['Excel', 'SQL', 'Data Analysis'],
  'experience': ['1 year'],
  'education': ['MBA'],
  'confidence_score': 0.95}]

In [23]:
final_schema_profiles = []


for profile in final_profiles:

    new_profile = {

        "candidate_id": profile["candidate_id"],

        "full_name": profile["full_name"],

        "emails": profile["emails"],

        "phones": profile["phones"],

        "skills": profile["skills"],

        "experience": profile["experience"],

        "education": profile["education"],

        "provenance": [
            {
                "source":"recruiter.csv",
                "matched":True
            },
            {
                "source":"resumes.json",
                "matched":True
            }
        ],

        "overall_confidence": profile["confidence_score"]

    }


    final_schema_profiles.append(new_profile)



final_schema_profiles

[{'candidate_id': 1,
  'full_name': 'Rahul Sharma',
  'emails': ['rahul@gmail.com'],
  'phones': ['+91-9876543210'],
  'skills': ['Python', 'SQL'],
  'experience': ['3 years'],
  'education': ['B.Tech'],
  'provenance': [{'source': 'recruiter.csv', 'matched': True},
   {'source': 'resumes.json', 'matched': True}],
  'overall_confidence': 0.99},
 {'candidate_id': 2,
  'full_name': 'Amit Verma',
  'emails': ['amit@gmail.com'],
  'phones': ['+91-9999999999'],
  'skills': ['Java', 'SQL'],
  'experience': ['2 years'],
  'education': ['BCA'],
  'provenance': [{'source': 'recruiter.csv', 'matched': True},
   {'source': 'resumes.json', 'matched': True}],
  'overall_confidence': 0.95},
 {'candidate_id': 4,
  'full_name': 'Priya Singh',
  'emails': ['priya@gmail.com'],
  'phones': ['+91-8888888888'],
  'skills': ['Excel', 'SQL', 'Data Analysis'],
  'experience': ['1 year'],
  'education': ['MBA'],
  'provenance': [{'source': 'recruiter.csv', 'matched': True},
   {'source': 'resumes.json', 'match

In [24]:
with open("output/candidate_profiles.json","w") as f:

    json.dump(
        final_schema_profiles,
        f,
        indent=4
    )


print("Final candidate_profiles.json created")

Final candidate_profiles.json created


In [25]:
with open("output/candidate_profiles.json") as f:

    final_output = json.load(f)


final_output

[{'candidate_id': 1,
  'full_name': 'Rahul Sharma',
  'emails': ['rahul@gmail.com'],
  'phones': ['+91-9876543210'],
  'skills': ['Python', 'SQL'],
  'experience': ['3 years'],
  'education': ['B.Tech'],
  'provenance': [{'source': 'recruiter.csv', 'matched': True},
   {'source': 'resumes.json', 'matched': True}],
  'overall_confidence': 0.99},
 {'candidate_id': 2,
  'full_name': 'Amit Verma',
  'emails': ['amit@gmail.com'],
  'phones': ['+91-9999999999'],
  'skills': ['Java', 'SQL'],
  'experience': ['2 years'],
  'education': ['BCA'],
  'provenance': [{'source': 'recruiter.csv', 'matched': True},
   {'source': 'resumes.json', 'matched': True}],
  'overall_confidence': 0.95},
 {'candidate_id': 4,
  'full_name': 'Priya Singh',
  'emails': ['priya@gmail.com'],
  'phones': ['+91-8888888888'],
  'skills': ['Excel', 'SQL', 'Data Analysis'],
  'experience': ['1 year'],
  'education': ['MBA'],
  'provenance': [{'source': 'recruiter.csv', 'matched': True},
   {'source': 'resumes.json', 'match

In [26]:
readme = """

# Candidate Data Transformer

## Problem Statement

This project transforms candidate data collected from multiple sources into one clean and unified candidate profile.

The system handles duplicate candidates, cleans data formats, merges information, and generates a standard JSON output.

---

## Data Sources

### Structured Source
- recruiter.csv

Contains:
- Candidate ID
- Name
- Email
- Phone
- Company
- Job Title


### Unstructured Source
- resumes.json

Contains:
- Skills
- Experience
- Education


---

## Processing Pipeline

Input Sources

        ↓

Data Extraction

        ↓

Normalization

        ↓

Duplicate Detection

        ↓

Profile Merging

        ↓

Confidence Calculation

        ↓

Final JSON Output


---

## Features

- Read multiple data sources
- Normalize phone numbers
- Normalize skills
- Detect duplicate candidates using email matching
- Merge candidate profiles
- Generate confidence score
- Export structured JSON output


---

## Output Schema

Example:

{
candidate_id,
full_name,
emails,
phones,
skills,
experience,
education,
provenance,
overall_confidence
}


---

## How To Run

1. Upload files:

recruiter.csv

resumes.json


2. Run notebook cells sequentially


3. Final output generated:

output/candidate_profiles.json


---

## Technologies Used

Python

Pandas

JSON


"""

with open("README.md","w") as f:
    f.write(readme)


print("README created")

README created


In [27]:
from IPython.display import display, Markdown

display(Markdown(open("README.md").read()))



# Candidate Data Transformer

## Problem Statement

This project transforms candidate data collected from multiple sources into one clean and unified candidate profile.

The system handles duplicate candidates, cleans data formats, merges information, and generates a standard JSON output.

---

## Data Sources

### Structured Source
- recruiter.csv

Contains:
- Candidate ID
- Name
- Email
- Phone
- Company
- Job Title


### Unstructured Source
- resumes.json

Contains:
- Skills
- Experience
- Education


---

## Processing Pipeline

Input Sources

        ↓

Data Extraction

        ↓

Normalization

        ↓

Duplicate Detection

        ↓

Profile Merging

        ↓

Confidence Calculation

        ↓

Final JSON Output


---

## Features

- Read multiple data sources
- Normalize phone numbers
- Normalize skills
- Detect duplicate candidates using email matching
- Merge candidate profiles
- Generate confidence score
- Export structured JSON output


---

## Output Schema

Example:

{
candidate_id,
full_name,
emails,
phones,
skills,
experience,
education,
provenance,
overall_confidence
}


---

## How To Run

1. Upload files:

recruiter.csv

resumes.json


2. Run notebook cells sequentially


3. Final output generated:

output/candidate_profiles.json


---

## Technologies Used

Python

Pandas

JSON




In [28]:
design_doc = """

# Candidate Data Transformer - Design Document


## 1. System Overview

The system collects candidate information from multiple sources and converts it into a single clean candidate profile.

Sources:

1. recruiter.csv (Structured Data)

2. resumes.json (Unstructured Resume Data)


---

## 2. Processing Pipeline


Input Sources

        |

        ↓

Data Extraction

        |

        ↓

Data Normalization

        |

        ↓

Duplicate Candidate Detection

        |

        ↓

Profile Merge

        |

        ↓

Confidence Score Calculation

        |

        ↓

Final Candidate JSON


---

## 3. Candidate Matching Logic


Candidates are matched using:

Primary key:

- Email address


Example:

Rahul Sharma
rahul@gmail.com


and


Rahul S.
rahul@gmail.com


are considered the same candidate.


---

## 4. Normalization Rules


Phone:

9876543210

98765-43210


Converted to:

+91-9876543210


Skills:

python programming

python

py


Converted to:

Python


---

## 5. Output Schema


candidate_id

full_name

emails

phones

skills

experience

education

provenance

overall_confidence


---

## 6. Edge Cases Handled


1. Duplicate candidates from different sources


2. Missing resume information


3. Different skill naming formats


4. Different phone number formats


---

## 7. Technology Used


Python

Pandas

JSON Processing


"""


with open("Design_Document.txt","w") as f:
    f.write(design_doc)


print("Design document created")

Design document created


In [29]:
import shutil
import os

# Create final project folder
os.makedirs("candidate-transformer", exist_ok=True)

# Copy files
shutil.copy("recruiter.csv", "candidate-transformer/")
shutil.copy("resumes.json", "candidate-transformer/")
shutil.copy("README.md", "candidate-transformer/")
shutil.copy("Design_Document.txt", "candidate-transformer/")

# Copy output folder
shutil.copytree("output", "candidate-transformer/output", dirs_exist_ok=True)

print("Project folder ready")

Project folder ready


In [30]:
shutil.make_archive(
    "candidate-transformer",
    "zip",
    "candidate-transformer"
)

print("Zip created")

Zip created


In [31]:
from google.colab import files

files.download("candidate-transformer.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
import os

os.listdir("output")

['final_profile.json', 'final_candidates.json', 'candidate_profiles.json']

In [33]:
from google.colab import files

files.download("output/candidate_profiles.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>